# Step 4b: Jev with the same information as Haiku and Sonnet
### Neeraj, MMAI 894 (router fine-tuning)

**Why:** in notebook 04, Jev was not told what Haiku and Sonnet were told. It got the request types, the tie-breaks and the escalation rules, but no worked examples, and its reason question had one-line definitions written by us instead of the prompt's own (for example, "an attack" without the five kinds of attack the prompt lists). Its weak reasons may come from that, so the comparison was not fair to Jev.

**Version 2 of Jev's questions** (`jev_questions_v2` in `baselines.py`): every question gets the team prompt's whole rules (its first paragraph and its Request types, Escalation and Language sections), the prompt's 6 worked examples with the answer to that question, and reason definitions copied from the prompt. Left out: only the lines about returning a JSON object and a confidence number, which do not apply to typed questions. One difference cannot be removed: Jev answers each question on its own, while Haiku answers all of them in one go.

**Same as notebook 04:** the same 160 pool chats, both inputs, 2 repeats, the same scoring. Haiku and Sonnet are not run again; their saved answers are read from the repo.

**What you need to do:** your OpenRouter key must be attached (Add-ons -> Secrets, tick `OPENROUTER_API_KEY` for this notebook). Accelerator None, Internet On, run all cells: about 10 minutes. **Expected cost well under $1** (Step 2 prints the estimate before the run; hard stop at $3). Download `results_04b.zip` and the notebook into `results/step04b`.

Everything shown is computed from saved answers; this text holds no results.

## Step 0: Get the latest code

In [ ]:
import importlib
import os
import shutil
import sys

REPO_URL = "https://github.com/neerajmehla28/northstar-finetuning"
REPO_DIR = "/kaggle/working/northstar-finetuning"

%cd /kaggle/working
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
sys.path.insert(0, "src")
importlib.invalidate_caches()
for name in [m for m in sys.modules if m.startswith("northstar")]:
    del sys.modules[name]

from northstar.finetune import baselines as B
assert hasattr(B, "jev_questions_v2"), "code missing: has the latest code been pushed to GitHub?"
assert os.path.exists("results/step04/preds"), "notebook 04's answers are missing from the repo"
!git log --oneline -3

## Step 1: The key, the data, and what Jev is asked
The key is read from Kaggle's secret store and never printed. The cell prints the start and the end of one version-2 question, so you can see what Jev is told.

In [ ]:
import json
import time

import numpy as np
import pandas as pd
try:
    from kaggle_secrets import UserSecretsClient
except ImportError:          # not on Kaggle
    UserSecretsClient = None

from northstar.finetune import protocol as P

API_KEY = None
try:                                    # Kaggle secret; without one, the saved answers are replayed for free
    API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
except Exception:
    pass

pool = P.pool(P.load_all())
RESULTS = "/kaggle/working/results_04b"
PREDS, CACHE = f"{RESULTS}/preds", f"{RESULTS}/cache"
VARIANTS = ("first_message", "full_conversation")
REPEATS = 2
BUDGET_USD = 3.0
client = B.Client(API_KEY, CACHE, budget_usd=BUDGET_USD)
if os.path.exists("results/step04b/cache"):            # the answers saved in the repo: reused, never paid for twice
    shutil.copytree("results/step04b/cache", CACHE, dirs_exist_ok=True)
print("API key:", "loaded" if API_KEY else "none, so the saved answers are replayed (free; stops at any call that was not saved)")
os.makedirs(RESULTS, exist_ok=True)

q2, q1 = B.jev_questions_v2(), B.jev_questions()
print("Characters of instructions per question, version 1 -> version 2:")
for k in q2:
    print(f"  {k:13s} {len(q1[k]['instructions']):5d} -> {len(q2[k]['instructions']):5d}")
print("\nThe reason question, version 2 (start and end):\n")
print(q2["reason"]["instructions"][:600], "\n   [...]\n", q2["reason"]["instructions"][-1100:])
print("\nReason definitions, version 2:")
for k, v in q2["reason"]["criteria"].items():
    print(f"  {k}: {v[:160]}{'...' if len(v) > 160 else ''}")
json.dump(dict(model=B.MODELS["jev2"], questions="jev_questions_v2", variants=VARIANTS, repeats=REPEATS, budget_usd=BUDGET_USD,
               date=time.strftime("%Y-%m-%d"), git_commit=os.popen("git rev-parse --short HEAD").read().strip()),
          open(f"{RESULTS}/run_config.json", "w"), indent=1)

## Step 2: Probe (3 chats) and cost estimate

In [ ]:
probe = B.run_variant(client, "jev2", pool, "first_message", 0, PREDS, limit=3)
display(probe[["id", "type", "esc", "language", "reason", "confidence", "latency_ms", "cost_usd", "fallback", "error"]])
assert not probe["fallback"].any(), "a probe call gave no usable answer: see the error column"
per_chat = probe["cost_usd"].mean()
estimate = per_chat * len(pool) * len(VARIANTS) * REPEATS * 1.15 + per_chat * 40
assert estimate < BUDGET_USD, f"estimate ${estimate:.2f} is above the limit"
print(f"Cost per chat: ${per_chat:.6f} (version 1 was about $0.000074) | estimated run: ${estimate:.2f} (limit ${BUDGET_USD:.0f})")

## Step 3: The full run
Both inputs, 2 repeats, every pool chat. Rerun the cell after an interruption: saved answers are not paid for twice.

In [ ]:
for v in VARIANTS:
    for rep in range(REPEATS):
        t0 = time.time()
        d = B.run_variant(client, "jev2", pool, v, rep, PREDS, workers=4)
        print(f"jev2 {v:17s} repeat {rep}: {len(d)} chats | fallbacks {int(d['fallback'].sum())} | ${d['cost_usd'].sum():.3f} | {time.time() - t0:.0f}s", flush=True)
print(f"New money spent in this session: ${client.spent:.2f}")

## Step 4: Speed, one call at a time
The same 40 chats as notebook 04 (first message), sent one after another; the first call is a warm-up and is left out.

In [ ]:
lp = B.latency_probe(client, "jev2", pool, "first_message", n=40).iloc[1:]
latency = pd.DataFrame([dict(model="jev2", chats=len(lp), median_ms=lp["latency_ms"].median(), p90_ms=lp["latency_ms"].quantile(0.9))])
latency.round(0).to_csv(f"{RESULTS}/latency.csv", index=False)
latency.round(0)

## Step 5: Jev version 2 against version 1, Haiku and Sonnet
Same chats, same scoring (combined answers; handoffs scored with the team's rules, as in notebook 05). Version 1 and the Claude models are read from notebook 04's saved answers.

In [ ]:
from northstar.finetune import router_options as O
from northstar.finetune import metrics as M

SOURCES = {"haiku": "results/step04/preds", "sonnet": "results/step04/preds", "jev": "results/step04/preds", "jev2": PREDS}
NAMES = {"haiku": "Haiku", "sonnet": "Sonnet", "jev": "Jev, version 1", "jev2": "Jev, version 2 (same information)"}
gold_esc = pool["escalation_label"].values == 1
attack = pool["attack_label"].values == 1
rows, handled = [], {}
for v in VARIANTS:
    for m, folder in SOURCES.items():
        reps = B.load_preds(folder, m, v, pool)
        llm = O.combine_llm(reps)
        s, routed = O.score(pool, O.option_llm_alone(llm))
        handled[(v, m)] = routed["handled_right"].values
        esc = llm["esc"].values
        rows.append({"input": v, "model": NAMES[m], "type right %": round(100 * s["type_accuracy"], 1),
                     "needs a person: caught (of 69)": int((esc & gold_esc).sum()), "false alarms (of 91)": int((esc & ~gold_esc).sum()),
                     "attacks called adversarial (of 31)": int((llm["reason"].values == "adversarial")[attack].sum()),
                     "right team (of 69)": s["right_team"], "attacks to right team (of 31)": s["attacks_right_team"],
                     "extra handoffs": s["extra_handoffs"], "handled right %": round(100 * s["handled_right"], 1),
                     "$ per 1,000 chats": round(float(np.mean([r["cost_usd"].mean() for r in reps])) * 1000, 3)})
table = pd.DataFrame(rows)
pairs = []
for v in VARIANTS:
    for m in ("jev2", "jev"):
        t = M.mcnemar_exact(handled[(v, m)], handled[(v, "haiku")])
        pairs.append(dict(input=v, model=NAMES[m], versus="Haiku", only_model_right=t["only_a_right"], only_haiku_right=t["only_b_right"], p_value=round(t["p_value"], 4)))
    t = M.mcnemar_exact(handled[(v, "jev2")], handled[(v, "jev")])
    pairs.append(dict(input=v, model=NAMES["jev2"], versus=NAMES["jev"], only_model_right=t["only_a_right"], only_haiku_right=t["only_b_right"], p_value=round(t["p_value"], 4)))
table.to_csv(f"{RESULTS}/jev_v2_comparison.csv", index=False)
pd.DataFrame(pairs).to_csv(f"{RESULTS}/jev_v2_paired.csv", index=False)
display(table.set_index(["input", "model"]))
display(pd.DataFrame(pairs).rename(columns={"only_haiku_right": "only_other_right"}))

## Step 6: Save the results
Download `results_04b.zip` (directly in `/kaggle/working`) and the notebook (File -> Download Notebook) into `results/step04b`.

In [ ]:
shutil.make_archive("/kaggle/working/results_04b", "zip", RESULTS)
print(f"results_04b.zip {os.path.getsize('/kaggle/working/results_04b.zip') / 1024:.0f} KB")

## What this notebook shows, and its limits
- **Shows:** how Jev does when it is told exactly what Haiku and Sonnet are told, on the same chats and scoring, and what that costs.
- **Limits:**
  - Jev still answers each question on its own (its reason does not see its own escalation answer); Haiku answers all four together.
  - The longer instructions are repeated for each of the four questions, so Jev reads more text per chat than Haiku; its price is per word read, so this costs more than version 1.
  - The team tuned the prompt on 30 of these chats, which helps every model that reads it, Jev version 2 included.
  - This is the second version of Jev's questions on the pool; Haiku's prompt had three team versions.